# Chess Coach Models: the lab's runs on a GPU

Runs the lab's own commands on this notebook's GPU: for each model in the plan, its dataset from
open Lichess data, its runs, and its leaderboard. Choose a GPU runtime (Runtime, Change runtime
type), set the plan in the first cell (a model and its runs, or the name of a plan in
`colab/plans/`), and run every cell.

The cells, in order: the plan; the GPU; the clone and its packages; the datasets; a check of every
run on a small sample; the full runs; the leaderboards; the two files to save. To train a second
plan on the same runtime, set the first cell, run it, and run every cell after the clone again:
the clone cell starts from an empty folder, so running it again would delete the datasets and the
runs already there.

The last cell packs two files: `/content/lab-results.zip`, the run records and leaderboards, and
`/content/lab-runs.zip`, every run's trained weights and test predictions. **Save both from the
Files panel before the runtime is released**: a runtime's disk goes with it. Unzip both at the
root of a clone of the repository (the weights land under `artifacts/runs/`, which git ignores),
then `lab eval <model>` for each model, `lab report`, commit and push.

The notebook holds no credentials and pushes nothing. `docs/compute.md` in the repository says
what runs where.

In [ ]:
# The plan: each model to run, and its runs. "" trains every run of a model's leaderboard, and
# "study:<name>" stands for every run of one of its studies. To add runs to a model that already
# has results, name them with gbm-hand among them, as in "gbm-hand,transformer-medium-long" or
# "gbm-hand,study:games": every other run is compared with gbm-hand trained on this machine.
# PLAN can also be the name of a plan kept in colab/plans/: PLAN = "studies" reads
# colab/plans/studies.yaml once the repository is cloned.
PLAN = {
    "puzzle-difficulty": "",
    "puzzle-difficulty-unsolved": "",
    "blunder-hazard": "",
    "think-time": "",
    "tactic-found": "",
    "practical-chances": "",
}
REF = "main"

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!nproc && free -g | head -2

In [ ]:
%cd /content
!rm -rf chess-coach-models
!git clone --quiet --depth 1 --branch {REF} https://github.com/Aneesh-Pothuru/chess-coach-models.git
%cd /content/chess-coach-models
!git log --oneline -1
# Colab ships Torch with CUDA. The lab's other needs are small, and --no-deps keeps Torch as it is.
# Colab's own Polars is too old for the tactics dataset: 1.35 cannot open its source files, which
# hold a JSON-typed column, and the build then hangs. The floor replaces it. pip warns that
# Colab's cudf-polars wants the older one; the lab does not use it.
!pip install -q python-chess zstandard lightgbm 'polars>=2.1' pyyaml
!pip install -q --no-deps -e .
!python -c "import torch, lightgbm, polars, chess, zstandard; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"

In [ ]:
# Each model's dataset, built once from open data and kept on this runtime. A model that already
# has results is given new runs only on the build its committed records name: a run on another
# build could not share its leaderboard, so this stops before any GPU time is spent on one.
import json
from pathlib import Path

import yaml

shell = get_ipython()
if isinstance(PLAN, str):
    PLAN = yaml.safe_load(Path(f"colab/plans/{PLAN}.yaml").read_text())
print("the plan:")
for model, runs in PLAN.items():
    print(f"  {model}: {runs or 'every run of its leaderboard'}")
tasks = {model: yaml.safe_load(Path(f"models/{model}/task.yaml").read_text()) for model in PLAN}
for dataset in sorted({task["dataset"] for task in tasks.values()}):
    shell.system(f"lab data {dataset}")
    if shell.user_ns.get("_exit_code", 0) != 0:
        raise SystemExit(f"the build of {dataset} failed")
for model, runs in PLAN.items():
    name = yaml.safe_load(Path(f"datasets/{tasks[model]['dataset']}.yaml").read_text())["name"]
    built = json.loads(Path(f"data/processed/{name}/manifest.json").read_text())["fingerprint"]
    records = [
        json.loads(path.read_text())
        for path in Path(f"models/{model}/results").glob("*.json")
        if path.name not in ("leaderboard.json", "analysis.json")
    ]
    named = {record["dataset"].get("fingerprint", "unnamed") for record in records}
    print(f"{model}: {name} is build {built[:12]}; its committed runs read {sorted(one[:12] for one in named) or 'none yet'}")
    if runs and named and named != {built}:
        raise SystemExit(
            f"{model}: this build of {name} is not the one its committed runs read. "
            f'Take {model} out of PLAN, or set its runs to "" to train every run on this build.'
        )

In [ ]:
# A check that every run in the plan works on this GPU: each model's runs on a small sample, a
# few minutes in all. It writes under artifacts/smoke/ and is never a result.
shell = get_ipython()
for model, runs in PLAN.items():
    command = f"lab all {model} --smoke" + (f" --runs {runs}" if runs else "")
    print(f"\n=== {command}", flush=True)
    shell.system(command)
    if shell.user_ns.get("_exit_code", 0) != 0:
        raise SystemExit(f"the smoke run of {model} failed: fix that before spending GPU time on the full runs")

In [ ]:
import os

os.makedirs("artifacts/logs", exist_ok=True)
shell = get_ipython()
for model, runs in PLAN.items():
    command = f"lab all {model}" + (f" --runs {runs}" if runs else "")
    print(f"\n=== {command}", flush=True)
    shell.system(f"bash -c 'set -o pipefail; {command} 2>&1 | tee artifacts/logs/{model}.log'")
    if shell.user_ns.get("_exit_code", 0) != 0:
        raise SystemExit(f"{command} failed: read artifacts/logs/{model}.log")

In [ ]:
from pathlib import Path

for model in PLAN:
    print(Path(f"models/{model}/results/leaderboard.md").read_text())

In [ ]:
!rm -f /content/lab-results.zip /content/lab-runs.zip
!zip -q -r /content/lab-results.zip models/*/results artifacts/logs
# The trained weights and each run's test predictions exist nowhere but on this runtime.
!zip -q -r /content/lab-runs.zip artifacts/runs
!ls -la /content/lab-results.zip /content/lab-runs.zip && sha256sum /content/lab-results.zip /content/lab-runs.zip
print("LAB RUNS DONE: save /content/lab-results.zip and /content/lab-runs.zip from the Files panel now")